# Tarea Práctica: Riesgos de Integridad en Entornos Distribuidos

## Contexto
Trabajas como Analista de Datos en un *E-commerce* de rápido crecimiento. El sistema de procesamiento de datos es distribuido: eventos de navegación y transacciones llegan continuamente desde microservicios separados (inventario, carrito, pasarela de pagos) a través de un sistema de mensajería (como Kafka).

Al ser una arquitectura distribuida (con retardos de red, reintentos automáticos y equipos de desarrollo separados), empiezan a aparecer comportamientos anómalos en los registros analíticos diarios.

## Objetivo
Tu objetivo no es la simple "limpieza por valores atípicos" (como hicimos en IoT), sino reparar **riesgos críticos de integridad técnica y lógica** en un *subset* representativo de los datos.

Debes abordar los siguientes problemas extraídos de la **UT3 - Capítulo 2**:
1.  **Duplicados y Replays:** El broker de mensajería reintenta envíos si falla la red, inyectando transacciones repetidas.
2.  **Estados Parciales:** Fallos en el pipeline de enriquecimiento hacen que algunas compras lleguen sin asociar al usuario (`user_id`).
3.  **Corrupción Lógica Silenciosa:** Un cambio silencioso de esquema en origen provoca que números lleguen como textos con formato europeo (ej: `"14,50 €"`), rompiendo los cálculos y devolviendo `NaN` tras conversiones fallidas.

### ¡IMPORTANTE! ⚠️
Como siempre, **justifica** por qué tomas cada decisión (¿borras un nulo? ¿rellenas? ¿te fías de qué columna para borrar duplicados?). Si delegas la justificación exclusivamente a una IA y no refleja tu entendimiento de la teoría, se notará y mas cuando te pregunte en clase.🥸 No me seas, y se curioso y pierde un poco el tiempo en amprender.


In [1]:
import pandas as pd
import numpy as np

# --- CÓDIGO DE GENERACIÓN DE DATOS (NO MODIFICAR) ---
# Tienes una muestra de 8 eventos de la última hora en la Capa Bronce (Raw).

data = {
    'event_id': ['EV-100', 'EV-101', 'EV-101', 'EV-102', 'EV-103', 'EV-104', 'EV-105', 'EV-106'],
    'timestamp': [
        '2026-03-01 14:00:00',
        '2026-03-01 14:05:00',
        '2026-03-01 14:05:00',  # Reintento exacto del EV-101
        '2026-03-01 14:10:00',
        '2026-03-01 14:12:00',
        '2026-03-01 14:15:00',
        '2026-03-01 14:10:00',  # Evento rezagado (old timestamp)
        '2026-03-01 14:20:00'
    ],
    'user_id': ['U-01', 'U-02', 'U-02', None, 'U-03', 'U-01', 'U-04', 'U-05'], # El EV-102 perdió la identidad del usuario
    'action': ['view', 'purchase', 'purchase', 'purchase', 'view', 'purchase', 'view', 'purchase'],
    'amount': [0.0, 50.5, 50.5, 120.0, 0.0, "25,99 €", 0.0, -10.0] # Falla esquema y valor negativo
}

df_events = pd.DataFrame(data)
print("--- Dataset Raw (Sucio) extraído de Data Lake ---")
display(df_events)

--- Dataset Raw (Sucio) extraído de Data Lake ---


,event_id,timestamp,user_id,action,amount
0,EV-100,2026-03-01 14:00:00,U-01,view,0.0
1,EV-101,2026-03-01 14:05:00,U-02,purchase,50.5
2,EV-101,2026-03-01 14:05:00,U-02,purchase,50.5
3,EV-102,2026-03-01 14:10:00,None,purchase,120.0
4,EV-103,2026-03-01 14:12:00,U-03,view,0.0
5,EV-104,2026-03-01 14:15:00,U-01,purchase,"25,99 €"
6,EV-105,2026-03-01 14:10:00,U-04,view,0.0
7,EV-106,2026-03-01 14:20:00,U-05,purchase,-10.0


---
### Ejercicio 1: Idempotencia y Duplicados por Reintentos (Replays)
*(Ref: UT3 punto 2.1c)*

**Problema:** En sistemas distribuidos, si el consumidor no confirma haber recibido la transacción `EV-101`, el productor la reenvía *"por si acaso"*. Si sumamos directamente los ingresos, estaríamos contando más dinero del real.

**Tarea:**
1. Detecta qué fila está duplicada, garantizando la idempotencia basándote exclusivamente en el identificador único de la transacción (`event_id`).
2. Elimina la fila fantasma manteniendo de forma segura la original.


### Justificacion Teorica - Ejercicio 1: Idempotencia y Duplicados por Reintentos

**Concepto aplicado (UT3, punto 2.1c):** El dossier identifica los **duplicados por reintentos** como uno de los riesgos de integridad mas frecuentes en sistemas distribuidos. En arquitecturas con brokers de mensajeria (Kafka, RabbitMQ), la garantia de entrega *at-least-once* significa que si el consumidor no confirma la recepcion del evento, el productor lo reenvio de forma automatica. El resultado es un evento identico inyectado dos veces en el Data Lake.

**Por que usar `event_id` como clave de idempotencia:** El `event_id` es el identificador unico de la transaccion asignado por el sistema origen. Es inmutable y no depende del timestamp (que en un reenvio puede ser distinto al original). El dossier menciona explicitamente el uso de **IDs unicos de evento** como mecanismo de idempotencia: si el sistema ve el mismo `event_id` dos veces, sabe que es un replay y puede descartarlo de forma segura. Usar el timestamp solo no seria valido porque dos compras legitimas distintas podrian ocurrir en el mismo segundo.

**Por que `keep='first'`:** La primera ocurrencia es el evento original que el productor envio correctamente. La segunda (con el mismo `event_id` pero timestamp ligeramente distinto) es el reenvio automatico del broker. Conservar la primera garantiza la coherencia con el sistema origen (source of truth).

In [2]:
# EJERCICIO 1: IDEMPOTENCIA Y DUPLICADOS POR REINTENTOS
# El broker de mensajeria garantiza at-least-once delivery:
# si el consumidor no confirma el ACK, el broker reenvía el mismo
# evento. Esto puede inyectar duplicados exactos (mismo event_id)
# en el Data Lake, inflando metricas criticas como ingresos totales.

# Paso 1: Detectar duplicados usando event_id como clave de idempotencia
# El event_id es el identificador unico e inmutable asignado por el sistema
# origen. Es la clave natural de deduplicacion. No usamos timestamp porque
# en un reenvio el timestamp puede ser ligeramente distinto al original.
duplicados_replay = df_events[df_events.duplicated(subset=['event_id'], keep='first')]
print("=== REPLAYS DETECTADOS (EVENTOS DUPLICADOS) ===")
print(f"Numero de replays: {len(duplicados_replay)}")
display(duplicados_replay)

# Paso 2: Eliminar el replay manteniendo la primera ocurrencia (el original)
# keep='first' conserva el evento original y descarta el reenvio del broker.
df_events = df_events.drop_duplicates(subset=['event_id'], keep='first')

print(f"\nFilas antes de deduplicar: 8")
print(f"Filas despues de deduplicar: {len(df_events)}")
print("\n=== DATASET TRAS ELIMINAR REPLAYS ===")
display(df_events)

=== REPLAYS DETECTADOS (EVENTOS DUPLICADOS) ===
Numero de replays: 1


,event_id,timestamp,user_id,action,amount
2,EV-101,2026-03-01 14:05:00,U-02,purchase,50.5



Filas antes de deduplicar: 8
Filas despues de deduplicar: 7

=== DATASET TRAS ELIMINAR REPLAYS ===


,event_id,timestamp,user_id,action,amount
0,EV-100,2026-03-01 14:00:00,U-01,view,0.0
1,EV-101,2026-03-01 14:05:00,U-02,purchase,50.5
3,EV-102,2026-03-01 14:10:00,None,purchase,120.0
4,EV-103,2026-03-01 14:12:00,U-03,view,0.0
5,EV-104,2026-03-01 14:15:00,U-01,purchase,"25,99 €"
6,EV-105,2026-03-01 14:10:00,U-04,view,0.0
7,EV-106,2026-03-01 14:20:00,U-05,purchase,-10.0


---
### Ejercicio 2: Estados Parciales y Registros Huérfanos
*(Ref: UT3 punto 2.1e)*

**Problema:** El evento `EV-102` es una compra real (`amount = 120.0`) pero se ha quedado en **estado parcial**. Existe, pero le falta el `user_id` asociado porque un microservicio que enriquece los datos de cliente estaba caído.

**Tarea:**
1. A diferencia del caso del "Edificio Inteligente" donde la falta de temperatura se podía interpolar, imputar un `user_id` a la ligera inventándolo puede arruinar métricas críticas como el Ticket Medio por Usuario o cruces de facturación.
2. Demuestra cómo identificarías las compras "huérfanas" (`action == 'purchase'` sin `user_id`).
3. Elimina o mueve ese registro a un dataframe de "cuarentena" justificando tu elección.


### Justificacion Teorica - Ejercicio 2: Estados Parciales y Registros Huerfanos

**Concepto aplicado (UT3, punto 2.1e):** El dossier describe los **estados parciales** (*partial writes / half-commits*) como uno de los problemas mas traiicioneros en sistemas distribuidos. Ocurren cuando una operacion se queda a medias: en este caso, el microservicio de enriquecimiento de clientes estaba caido cuando llego la compra EV-102, por lo que el `user_id` nunca se asocio al evento. El registro existe pero esta incompleto, lo que el dossier llama un **registro huerfano**: tiene datos de transaccion validos pero le falta su entidad padre (el usuario).

**Por que NO imputar el `user_id`:** A diferencia de la temperatura en un sensor, donde la media del propio sensor es una estimacion razonable, inventar un `user_id` seria catastrofico para metricas de negocio criticas: el Ticket Medio por Usuario, el historial de compras, los cruces de facturación o la deteccion de fraude. Asignar EV-102 a un usuario incorrecto introduciria una inconsistencia logica silenciosa en toda la cadena analitica.

**Por que enviar a cuarentena y no descartar:** La cuarentena sigue el patron *dead-letter queue* del dossier: separamos el registro problematico con trazabilidad para que el equipo de ingenieria pueda recuperarlo cuando el microservicio se recupere y retroalimentar el pipeline. Descartarlo silenciosamente seria una perdida de datos real (la compra de 120 euros existio).

In [3]:
# EJERCICIO 2: ESTADOS PARCIALES Y REGISTROS HUERFANOS
# El microservicio de enriquecimiento de cliente estaba caido
# cuando llego EV-102, dejando el evento sin user_id.
# En sistemas distribuidos esto se conoce como 'partial write' o
# 'half-commit': la operacion se quedo a medias y el registro
# existe pero le falta su entidad padre. No se puede imputar
# un user_id arbitrario: destruiria metricas criticas de negocio.

# Paso 1: Identificar compras huerfanas
# Condicion: action='purchase' Y user_id es nulo.
# Solo aplicamos a 'purchase' porque 'view' no necesita user_id
# para ser analitico (los views pueden ser de usuarios anonimos).
mascara_huerfana = (df_events['action'] == 'purchase') & (df_events['user_id'].isna())
compras_huerfanas = df_events[mascara_huerfana]
print("=== COMPRAS HUERFANAS DETECTADAS ===")
print(f"Registros sin user_id en compras: {len(compras_huerfanas)}")
display(compras_huerfanas)

# Paso 2: Mover a cuarentena con motivo del fallo
# Razon: no descartamos porque la compra es real (120.0),
# pero tampoco la incluimos en analitica sin user_id.
# La cuarentena (patron dead-letter) permite recuperarla
# cuando el microservicio vuelva a estar disponible.
df_cuarentena_parcial = compras_huerfanas.copy()
df_cuarentena_parcial['motivo_cuarentena'] = 'estado_parcial: user_id ausente por microservicio caido'

# Paso 3: Retirar del dataset principal
df_events = df_events[~mascara_huerfana].copy()

print(f"\nFilas enviadas a cuarentena: {len(df_cuarentena_parcial)}")
print(f"Filas validas restantes: {len(df_events)}")
print("\n=== DATASET TRAS AISLAR HUERFANOS ===")
display(df_events)
print("\n=== CUARENTENA ESTADOS PARCIALES ===")
display(df_cuarentena_parcial)

=== COMPRAS HUERFANAS DETECTADAS ===
Registros sin user_id en compras: 1


,event_id,timestamp,user_id,action,amount
3,EV-102,2026-03-01 14:10:00,None,purchase,120.0



Filas enviadas a cuarentena: 1
Filas validas restantes: 6

=== DATASET TRAS AISLAR HUERFANOS ===


,event_id,timestamp,user_id,action,amount
0,EV-100,2026-03-01 14:00:00,U-01,view,0.0
1,EV-101,2026-03-01 14:05:00,U-02,purchase,50.5
4,EV-103,2026-03-01 14:12:00,U-03,view,0.0
5,EV-104,2026-03-01 14:15:00,U-01,purchase,"25,99 €"
6,EV-105,2026-03-01 14:10:00,U-04,view,0.0
7,EV-106,2026-03-01 14:20:00,U-05,purchase,-10.0



=== CUARENTENA ESTADOS PARCIALES ===


,event_id,timestamp,user_id,action,amount,motivo_cuarentena
3,EV-102,2026-03-01 14:10:00,None,purchase,120.0,estado_parcial: user_id ausente por microservi...


---
### Ejercicio 3: Cambio de Esquema y Corrupción Silenciosa Lógica
*(Ref: UT3 punto 2.2c y 2.1b)*

**Problema:** Uno de los equipos front-end hizo una actualización silenciosa y empezó a enviar los importes concatenados con texto español (`"25,99 €"`) en la compra `EV-104`.
Al mezclarse números y Strings `amount` pasa a ser una columna `Object`. También hay un valor sin sentido (`-10.0`).

**Tarea:**
1. Muestra el tipo de dato (`dtypes`) actual de la columna `amount`.
2. Repara la columna: limpia el texto (cambia la coma por punto y quita el " €") solo en las celdas que sean texto, o fuerza a que toda la columna se convierta a `float`. `pd.to_numeric(errors='coerce')` puede ser tu gran aliado táctico.
3. Las compras no pueden tener costes negativos. Localiza el valor numérico contaminado y arréglalo pasándolo a 0 o borrando la fila.

### Justificacion Teorica - Ejercicio 3: Cambio de Esquema y Corrupcion Silenciosa Logica

**Concepto aplicado (UT3, puntos 2.2c y 2.1b):** El dossier identifica dos riesgos combinados aqui. El primero es el **cambio de esquema no controlado** (punto 2.2c): el equipo front-end actualizo silenciosamente el formato del campo `amount` de float numerico a string con formato europeo. Al no existir un contrato de datos ni un Schema Registry, el cambio llego sin avisar al pipeline y corrompio logicamente la columna. El segundo es la **corrupcion de datos** (punto 2.1b): el tipo de dato parece valido (es un string bien formado) pero el significado es incorrecto para operaciones numericas. Esto es un fallo silencioso tipico: el pipeline no rompe, pero los calculos devuelven NaN.

**Por que usar `str.replace` + `pd.to_numeric(errors='coerce')`:** Solo las celdas con formato europeo necesitan limpieza de texto; las numericas ya son correctas. Combinamos la limpieza selectiva de texto con `pd.to_numeric(errors='coerce')` para convertir la columna completa a float de forma robusta: si alguna celda no se puede convertir (corrupciones no anticipadas) devuelve NaN en lugar de romper el pipeline.

**Por que eliminar el valor negativo (-10.0):** Las compras en un e-commerce no pueden tener importe negativo por definicion de negocio. Un -10.0 en `amount` es un valor imposible en el dominio (precision fallida, posiblemente un bug del productor). Lo tratamos como cuarentena para no perder la trazabilidad del evento.

In [5]:
# EJERCICIO 3: CAMBIO DE ESQUEMA Y CORRUPCION SILENCIOSA LOGICA
# El equipo front-end cambio silenciosamente el tipo de 'amount':
# de float numerico a string con formato europeo (ej: '25,99 EUR').
# Sin un Schema Registry o Data Contract, este cambio llega
# silenciosamente y provoca un fallo silencioso: el pipeline no falla
# pero los calculos devuelven NaN al intentar operar con strings.

# Paso 1: Diagnosticar el tipo de dato actual de amount
print("=== DIAGNOSTICO: TIPO DE DATO DE 'amount' ===")
print(f"Tipo actual: {df_events['amount'].dtype}")
print("Valores de amount antes de reparar:")
print(df_events[['event_id', 'amount']].to_string())

# Paso 2: Reparar la columna - limpieza selectiva + conversion robusta
# Aplicamos limpieza solo a strings: sustituimos coma por punto decimal
# y quitamos el simbolo de euro. Luego convertimos toda la columna con
# pd.to_numeric(errors='coerce'): si hay corrupcion no anticipada,
# produce NaN en lugar de romper el pipeline (gestion defensiva).
df_events['amount'] = df_events['amount'].apply(
    lambda x: str(x).replace(',', '.').replace('\u20ac', '').strip() if isinstance(x, str) else x
)
df_events['amount'] = pd.to_numeric(df_events['amount'], errors='coerce')

print(f"\nTipo despues de reparar: {df_events['amount'].dtype}")
print(df_events[['event_id', 'amount']].to_string())

# Paso 3: Cuarentenar el valor negativo (-10.0)
# Las compras no pueden tener importe negativo: es un valor
# de dominio imposible en el negocio. Lo enviamos a cuarentena
# (no descarte silencioso) para mantener trazabilidad del evento.
mascara_negativa = df_events['amount'] < 0
df_cuarentena_negativo = df_events[mascara_negativa].copy()
df_cuarentena_negativo['motivo_cuarentena'] = 'importe_negativo: valor de dominio imposible'
df_events = df_events[~mascara_negativa].copy()

print(f"\nRegistros con importe negativo (cuarentena): {len(df_cuarentena_negativo)}")
print(f"Filas validas restantes: {len(df_events)}")
print("\n=== DATASET FINAL TRAS REPARAR ESQUEMA ===")
display(df_events)

=== DIAGNOSTICO: TIPO DE DATO DE 'amount' ===
Tipo actual: object
Valores de amount antes de reparar:
  event_id   amount
0   EV-100      0.0
1   EV-101     50.5
4   EV-103      0.0
5   EV-104  25,99 €
6   EV-105      0.0
7   EV-106    -10.0

Tipo despues de reparar: float64
  event_id  amount
0   EV-100    0.00
1   EV-101   50.50
4   EV-103    0.00
5   EV-104   25.99
6   EV-105    0.00
7   EV-106  -10.00

Registros con importe negativo (cuarentena): 1
Filas validas restantes: 5

=== DATASET FINAL TRAS REPARAR ESQUEMA ===


,event_id,timestamp,user_id,action,amount
0,EV-100,2026-03-01 14:00:00,U-01,view,0.00
1,EV-101,2026-03-01 14:05:00,U-02,purchase,50.50
4,EV-103,2026-03-01 14:12:00,U-03,view,0.00
5,EV-104,2026-03-01 14:15:00,U-01,purchase,25.99
6,EV-105,2026-03-01 14:10:00,U-04,view,0.00


---
### Resultado Final Consolidado
Prueba a sumar todos los `amount` de la columna de las compras resultantes. Tu contabilidad debería ser perfecta ahora.

In [6]:
# RESULTADO FINAL CONSOLIDADO
# Resumen del pipeline de integridad aplicado (UT3, capitulo 2):
# 1. Idempotencia: eliminado 1 replay de EV-101 (duplicado por reintento del broker)
# 2. Estado Parcial: EV-102 enviado a cuarentena (compra sin user_id por microservicio caido)
# 3. Cambio de Esquema: reparado '25,99 EUR' -> 25.99 float en EV-104
# 4. Valor de dominio imposible: EV-106 (-10.0) enviado a cuarentena
# La contabilidad resultante es integra y auditable.

print("=====================================================")
print(" PIPELINE DE INTEGRIDAD COMPLETADO - RESUMEN FINAL")
print("=====================================================")
print(f"Eventos originales (dataset raw):       8")
print(f"Eventos tras eliminar replay (EV-101):  7")
print(f"Eventos tras aislar huerfano (EV-102):  6")
print(f"Eventos tras reparar esquema + filtrar: {len(df_events)}")
print(f"Eventos en cuarentena (parciales):      {len(df_cuarentena_parcial)}")
print(f"Eventos en cuarentena (negativos):      {len(df_cuarentena_negativo)}")
print("")

# Verificacion contable: suma de amount en compras validas
compras_finales = df_events[df_events['action'] == 'purchase']
total_ingresos = compras_finales['amount'].sum()
print("=== COMPRAS VALIDAS FINALES ===")
display(compras_finales[['event_id', 'user_id', 'action', 'amount']])
print(f"\nTotal de ingresos verificados: {total_ingresos:.2f} EUR")
print("(Breakdown: EV-101=50.5 + EV-103=0.0 + EV-104=25.99 + EV-105=0.0)")
print("")
print("=== DATASET FINAL LIMPIO (TODOS LOS EVENTOS) ===")
display(df_events)
print(f"\nNulos restantes en dataset: {df_events.isnull().sum().sum()}")
print(f"Tipo de amount: {df_events['amount'].dtype}")

 PIPELINE DE INTEGRIDAD COMPLETADO - RESUMEN FINAL
Eventos originales (dataset raw):       8
Eventos tras eliminar replay (EV-101):  7
Eventos tras aislar huerfano (EV-102):  6
Eventos tras reparar esquema + filtrar: 5
Eventos en cuarentena (parciales):      1
Eventos en cuarentena (negativos):      1

=== COMPRAS VALIDAS FINALES ===


,event_id,user_id,action,amount
1,EV-101,U-02,purchase,50.50
5,EV-104,U-01,purchase,25.99



Total de ingresos verificados: 76.49 EUR
(Breakdown: EV-101=50.5 + EV-103=0.0 + EV-104=25.99 + EV-105=0.0)

=== DATASET FINAL LIMPIO (TODOS LOS EVENTOS) ===


,event_id,timestamp,user_id,action,amount
0,EV-100,2026-03-01 14:00:00,U-01,view,0.00
1,EV-101,2026-03-01 14:05:00,U-02,purchase,50.50
4,EV-103,2026-03-01 14:12:00,U-03,view,0.00
5,EV-104,2026-03-01 14:15:00,U-01,purchase,25.99
6,EV-105,2026-03-01 14:10:00,U-04,view,0.00



Nulos restantes en dataset: 0
Tipo de amount: float64
